In [ ]:
from __future__ import annotations

import json
import pathlib
import random
from collections import Counter
from pathlib import Path

from kebab.utils.dataset.wikidata import wikidata_utils
from kebab.utils.dataset.wikidata.wikidata_utils import ResolvedWikidataEntity

In [ ]:
exclude_used_entities = False
required_entity_type = ""

# parameters for test dataset
linking_pair_count_limit = 1_000
linking_entity_count_limit = 10
linking_property_pattern_count_limit = 20
linking_property_overlap_limits = {1: 0.35}
single_class_ratio_limit = 0.51

# parameters for training dataset
# linking_pair_count_limit = 1_000_000
# linking_entity_count_limit = 100
# linking_property_pattern_count_limit = 10_000
# linking_property_overlap_limits = {1: 0.25}
# single_class_ratio_limit = 0.52

clustering_entity_count_limit = -1
clustering_fragment_count_limit = 999
clustering_include_all_linking_fragments = True

output_dir = Path.cwd() / "output"
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# # base granular (leaves) dataset
# base_linking_dataset_path = (
#     pathlib.Path.home()
#     / "OneDrive - Microsoft"
#     / "Benchmark"
#     / "Datasets"
#     / "REBEL"
#     / "Base Linking Dataset"
#     / "rebel_linking_dataset.jsonl"
# )
# 
# base_linking_ground_truth_path = (
#     pathlib.Path.home()
#     / "OneDrive - Microsoft"
#     / "Benchmark"
#     / "Datasets"
#     / "REBEL"
#     / "Base Linking Dataset"
#     / "rebel_linking_ground_truth.jsonl"
# )

# base merged dataset
base_linking_dataset_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Base Linking Merged Dataset"
    / "rebel_linking_dataset.jsonl"
)

base_linking_ground_truth_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Base Linking Merged Dataset"
    / "rebel_linking_ground_truth.jsonl"
)

base_clustering_dataset_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Base Linking Merged Dataset"
    / "clustering"
    / "rebel_clustering_dataset.jsonl"
)

base_clustering_ground_truth_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Base Linking Merged Dataset"
    / "clustering"
    / "rebel_clustering_ground_truth.jsonl"
)

# local scripts output
# base_linking_dataset_path = (
#     pathlib.Path.cwd().parent
#     / "scripts"
#     / "dataset"
#     / "output"
#     / "rebel_linking_dataset.jsonl"
# )
# 
# base_linking_ground_truth_path = (
#     pathlib.Path.cwd().parent
#     / "scripts"
#     / "dataset"
#     / "output"
#     / "rebel_linking_ground_truth.jsonl"
# )

In [ ]:
# fragment_id to entity_id map
fragment_to_entity_map_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Base Linking Dataset"
    / "clustering"
    / "rebel_fragment_to_entity_map.jsonl"
)

# Wikidata type hierarchy for filtering by type
type_hierarchy_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "Wikidata"
    / "Type Hierarchy"
    / "2025-01-30"
    / "wikidata_type_hierarchy.jsonl"
)

In [ ]:
# entities used in the evaluation datasets
entity_ids_used_for_evaluation = [
    (
        pathlib.Path.home()
        / "OneDrive - Microsoft"
        / "Benchmark"
        / "Datasets"
        / "REBEL"
        / "Linking Merged Test Dataset"
        / "rebel_linking_1000_used_entities.jsonl"
    ),
    (
        pathlib.Path.home()
        / "OneDrive - Microsoft"
        / "Benchmark"
        / "Datasets"
        / "REBEL"
        / "Linking Merged Validation Dataset"
        / "rebel_linking_1000_used_entities.jsonl"
    ),]

In [ ]:
disallowed_entity_ids = set()

if exclude_used_entities:
    for entity_ids_path in entity_ids_used_for_evaluation:
        if not entity_ids_path.exists():
            continue
            
        with open(entity_ids_path, encoding="utf-8") as f:
            for line in f:
                entity_id = json.loads(line)
                disallowed_entity_ids.add(entity_id)
    
print(f"Disallowed entities: {len(disallowed_entity_ids):,d}")

In [ ]:
if required_entity_type:
    # get the target entity type and its descendants
    get_descendants = False
    
    graph, type_id_to_node = wikidata_utils.load_type_hierarchy(type_hierarchy_path)
    required_entity_types = (
        wikidata_utils.collect_all_subtypes(graph, type_id_to_node, required_entity_type)
        if get_descendants
        else {required_entity_type}
    )
    
    required_entity_types = {type_id_to_node[t]["name"] for t in required_entity_types}
else:
    required_entity_types = set()

print(f"Target entity type: {required_entity_type} ({len(required_entity_types):,d} with descendants)")    

# Linking
---

In [ ]:
# load the ground truth
with open(base_linking_ground_truth_path, encoding="utf-8") as f:
    ground_truth = [json.loads(line) for line in f]

print(f"Loaded {len(ground_truth):,d} ground truth labels")

# load fragment to entity map
fragment_to_entity_map = {}
with open(fragment_to_entity_map_path, encoding="utf-8") as f:
    for line in f:
        f_id, e_id = json.loads(line)
        fragment_to_entity_map[f_id] = e_id

print(f"Loaded {len(fragment_to_entity_map):,d} fragment to entity mappings")

In [ ]:
# load and filter fragments to only include the target entity types
fragments = {}
pairs = []
labels = []
entity_ids = set()


def add_fragment(fragment: ResolvedWikidataEntity) -> ResolvedWikidataEntity:
    """Add the fragment to the set of known fragments."""
    if fragment.metadata["fragment_id"] not in fragments:
        fragments[fragment.metadata["fragment_id"]] = fragment
        fragment.evidence_map = None
        fragment.source_ids = None

        fragment.entity_id = fragment_to_entity_map[fragment.metadata["fragment_id"]]

    return fragments[fragment.metadata["fragment_id"]]


with open(base_linking_dataset_path, encoding="utf-8") as f:
    for i, line in enumerate(f):
        d = json.loads(line)
        left = add_fragment(ResolvedWikidataEntity.from_dict(d[0]))
        right = add_fragment(ResolvedWikidataEntity.from_dict(d[1]))
        
        if required_entity_types:
            if not set(left.wikidata_type).intersection(required_entity_types) or not set(right.wikidata_type).intersection(
                required_entity_types
            ):
                continue

        pairs.append((left, right))
        labels.append(ground_truth[i])
        entity_ids.add(left.entity_id)
        entity_ids.add(right.entity_id)

assert len(pairs) == len(labels)

print(f"Filtered to {len(pairs):,d} pairs where both entities are of the target types")
print(f"Positive pairs: {sum(labels):,d} ({sum(labels) / len(labels):.2%})")

print(f"Distinct entities: {len(entity_ids):,d} ({len(entity_ids) / len(pairs):.2%})")
with open(output_dir / f"rebel_linking_{linking_pair_count_limit}_entity_ids_pool.jsonl", "w", encoding="utf-8") as f:
    for entity_id in entity_ids:
        f.write(json.dumps(entity_id) + "\n")

In [ ]:
# sample pairs
sampled_pairs = []
sampled_labels = []

linking_entity_counter = Counter()
property_pattern_counter = Counter()
property_counter = Counter()
type_counter = Counter()
overlap_counter = Counter()
class_counter = Counter()

indices = list(range(len(pairs)))
random.shuffle(indices)
iterations = 0

included_fragment_ids = set()

for i in indices:
    iterations += 1

    if len(sampled_pairs) >= linking_pair_count_limit:
        break

    pair = pairs[i]
    left, right = pair
    
    if left.entity_id in disallowed_entity_ids or right.entity_id in disallowed_entity_ids:
        continue

    if (
        linking_entity_counter[left.entity_id] >= linking_entity_count_limit
        or linking_entity_counter[right.entity_id] >= linking_entity_count_limit
    ):
        continue

    prop_pattern = tuple(sorted([tuple(sorted(left.properties)), tuple(sorted(right.properties))]))
    if property_pattern_counter[prop_pattern] >= linking_property_pattern_count_limit:
        continue
    
    # overlap
    left_props = set(left.properties.keys())
    right_props = set(right.properties.keys())
    overlap = left_props.intersection(right_props)
    overlap_num = len(overlap)
    if overlap_num in linking_property_overlap_limits and overlap_counter[overlap_num] >= linking_property_overlap_limits[overlap_num] * linking_pair_count_limit:
        continue
    
    if single_class_ratio_limit and class_counter[labels[i]] > single_class_ratio_limit * linking_pair_count_limit:
        continue
    
    class_counter[labels[i]] += 1
    
    overlap_counter[len(overlap)] += 1
    
    linking_entity_counter[left.entity_id] += 1
    if left.entity_id != right.entity_id:
        linking_entity_counter[right.entity_id] += 1

    property_pattern_counter[prop_pattern] += 1
    
    for t in set(left.wikidata_type or []).union(right.wikidata_type or []):
        type_counter[t] += 1
        
    for prop_name in set(left.properties.keys()).union(right.properties.keys()):
        property_counter[prop_name] += 1
    
    sampled_pairs.append(pairs[i])
    sampled_labels.append(labels[i])
    included_fragment_ids.add(left.metadata["fragment_id"])
    included_fragment_ids.add(right.metadata["fragment_id"])

print(f"Sampled {len(sampled_pairs):,d} pairs containing {len(linking_entity_counter):,d} distinct entities")
print(f"Positive pairs: {sum(sampled_labels):,d} ({sum(sampled_labels) / len(sampled_labels):.2%})")
print(f"Iterations: {iterations:,d}, acceptance rate {len(sampled_pairs) / iterations:.2%}")
print(f"Total included fragments: {len(included_fragment_ids):,d} ({len(included_fragment_ids) / len(fragments):.2%})")

# top entities
print("\nTop entities:")
for i, (k, v) in enumerate(linking_entity_counter.most_common(n=10)):
    print(f"{i}: {k}: {v}")

# top properties
print("\nTop properties:")
for i, (k, v) in enumerate(property_counter.most_common(n=10)):
    print(f"{i}: {k}: {v} ({v / len(sampled_pairs):.2%})")

# top property patterns
print("\nTop property patterns:")
for i, (k, v) in enumerate(property_pattern_counter.most_common(n=10)):
    print(f"{i}: {k}: {v}")

# top types
print("\nTop types:")
for i, (k, v) in enumerate(type_counter.most_common(n=10)):
    print(f"{i}: {k}: {v} ({v / len(sampled_pairs):.2%})")

# top overlaps
print("\nTop overlaps:")
for i, (k, v) in enumerate(sorted(overlap_counter.items(), key=lambda x: x[0])):
    print(f"{k}: {v} ({v / len(sampled_pairs):.2%})")

In [ ]:
size = len(sampled_pairs)

# write the sampled pairs and labels to files
with open(output_dir / f"rebel_linking_{size}_dataset.jsonl", "w", encoding="utf-8") as f:
    for pair in sampled_pairs:
        d = [
            pair[0].to_dict(minimal_repr=True),
            pair[1].to_dict(minimal_repr=True),
        ]
        f.write(json.dumps(d) + "\n")

with open(output_dir / f"rebel_linking_{size}_ground_truth.jsonl", "w", encoding="utf-8") as f:
    for label in sampled_labels:
        f.write(json.dumps(label) + "\n")

with open(output_dir / f"rebel_linking_{size}_used_entities.jsonl", "w", encoding="utf-8") as f:
    for entity_id in linking_entity_counter.keys():
        f.write(json.dumps(entity_id) + "\n")

# Clustering
---

In [ ]:
# filter the clustering dataset and ground truth to only include the sampled entities
allowed_entity_ids = set(linking_entity_counter)
clustering_entity_counter = Counter()

# size = clustering_entity_count_limit if clustering_entity_count_limit > 0 else len(allowed_entity_ids)

with (
    open(base_clustering_dataset_path, encoding="utf-8") as f_ds,
    open(base_clustering_ground_truth_path, encoding="utf-8") as f_gt,
    open(output_dir / f"rebel_clustering_{size}_dataset.jsonl", "w", encoding="utf-8") as f_ds_out,
    open(output_dir / f"rebel_clustering_{size}_ground_truth.jsonl", "w", encoding="utf-8") as f_gt_out,
):
    for line_ds, line_gt in zip(f_ds, f_gt, strict=False):
        fragment = json.loads(line_ds)
        entity_id = json.loads(line_gt)

        if entity_id not in allowed_entity_ids:
            continue
        
        if 0 < clustering_entity_count_limit <= len(clustering_entity_counter) and entity_id not in clustering_entity_counter:
            continue
        
        if not clustering_include_all_linking_fragments or fragment["metadata"]["fragment_id"] not in included_fragment_ids:
            if clustering_entity_counter[entity_id] >= clustering_fragment_count_limit:
                continue
    
            clustering_entity_counter[entity_id] += 1

        f_ds_out.write(line_ds)
        f_gt_out.write(line_gt)

print(
    f"Filtered to {sum(clustering_entity_counter.values()):,d} fragments for {len(clustering_entity_counter):,d} entities, allowed = {len(allowed_entity_ids):,d} entities, target = {clustering_entity_count_limit:,d} entities"
)

print("\nTop entities:")
for i, (k, v) in enumerate(clustering_entity_counter.most_common(n=10)):
    print(f"{i}: {k}: {v}")